# Customer Retention Cohort Analysis


This notebook analyzes customer retention behavior using cohort analysis.

Key goals:
- Understand how long customers stay active after first purchase
- Measure retention rates across cohorts
- Identify drop-off patterns
- Derive actionable business insights to improve retention

## Data Preparation

We prepare data for cohort analysis by:
1. Converting dates to monthly format
2. Assigning cohort (first purchase month)
3. Calculating time difference (cohort index)

In [ ]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

#loading the dataset

customers = pd.read_csv('../data/raw/customers.csv')
sessions = pd.read_csv('../data/raw/sessions.csv')
cart = pd.read_csv('../data/raw/cart_events.csv')
orders = pd.read_csv('../data/raw/orders.csv')
products = pd.read_csv('../data/raw/products.csv')

In [ ]:


# Convert to datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Extract month
orders['order_month'] = orders['order_date'].dt.to_period('M')

# Cohort month = first purchase month
orders['cohort_month'] = orders.groupby('customer_id')['order_date'] \
                               .transform('min') \
                               .dt.to_period('M')

# Cohort index (months since first purchase)
orders['cohort_index'] = (orders['order_month'] - orders['cohort_month']) \
                         .apply(lambda x: x.n)

## Cohort Table

We calculate number of unique customers per cohort over time.

In [ ]:
cohort_data = orders.groupby(['cohort_month', 'cohort_index'])['customer_id'] \
                    .nunique() \
                    .reset_index()

cohort_pivot = cohort_data.pivot(index='cohort_month',
                                 columns='cohort_index',
                                 values='customer_id')

# Cohort size (Month 0)
cohort_size = cohort_pivot.iloc[:, 0]

# Retention matrix
retention = cohort_pivot.divide(cohort_size, axis=0)

## Retention Heatmap

This heatmap shows retention percentage across cohorts over time.

- Rows → Cohort (first purchase month)
- Columns → Months since acquisition
- Values → % of customers retained

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(retention, annot=True, fmt=".0%", cmap="YlGnBu")
plt.title("Customer Retention Cohort Analysis")
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort Month")
plt.show()

## Key Retention Metrics

We define key KPIs:

- Month 1 Retention → % users returning after 1 month
- Month 3 Retention → medium-term retention
- Average Retention → overall retention performance
- Churn Rate → % customers lost
- Repeat Purchase Rate → returning customers

In [ ]:
# Month 1 retention
month1_retention = retention[1].mean()

# Month 3 retention
month3_retention = retention[3].mean()

# Average retention
avg_retention = retention.mean().mean()

# Churn rate
churn_rate = 1 - month1_retention

# Repeat purchase rate
repeat_customers = orders.groupby('customer_id')['order_id'].count()
repeat_rate = (repeat_customers > 1).mean()

kpi_table = pd.DataFrame({
    "Metric": ["Month 1 Retention", "Month 3 Retention", "Avg Retention", "Churn Rate", "Repeat Purchase Rate"],
    "Value": [month1_retention, month3_retention, avg_retention, churn_rate, repeat_rate]
})

kpi_table

## Key Insights

- Significant drop in retention after first month
- Retention stabilizes after month 2–3
- Later cohorts show slight improvement (if applicable)
- Majority of customers are one-time buyers

## Business Interpretation

- High early churn suggests weak onboarding or low engagement
- Stable long-term retention indicates a loyal customer base
- Improving first-month retention can significantly increase revenue

## Recommendations

- Improve onboarding experience (emails, offers)
- Encourage second purchase with targeted discounts
- Personalize recommendations after first order
- Retarget churned users within first 30 days